In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, to_date, upper, when

In [0]:
gl_df = spark.table("01_global_tech_bronze.raw_tables.general_ledgers_raw")
accounts_df = spark.table("02_global_tech_silver.transformed_tables.transformed_accounts")

transformed_gl_df = (
    gl_df
    .withColumn("entry_date", to_date(col("entry_date"), "dd-MM-yyyy HH:mm"))
    .withColumn("posting_date", to_date(col("posting_date"), "dd-MM-yyyy HH:mm"))
    .withColumnRenamed("status", "gl_status")
    .withColumn("gl_sk", monotonically_increasing_id())
    .select(
        "gl_sk", "gl_id", "company_id", "account_id", "department_id",
        "entry_date", "posting_date", "fiscal_year", "fiscal_period",
        "transaction_type", "reference_number", "description",
        "debit_amount", "credit_amount", "created_by", "approved_by", "gl_status"
    )
)

# transformed_gl_df = (
#     gl_df.alias("gl")
#     .join(accounts_df.alias("acc"), col("gl.account_id") == col("acc.account_id"), "left")
#     .select(col("gl.*"), col("acc.account_type"))
# )

In [0]:
transformed_gl_df.write.mode("overwrite").saveAsTable("02_global_tech_silver.transformed_tables.transformed_general_ledgers")